# Uso de script de modularización — bielección (bloque largo t-2 -> t)

Misma estructura que `ventana_t-1/01.3_lasso_voto_exit.ipynb`, sobre
`data/tfi_data/panel_ventanas_bieleccion.csv`. Variable a predecir:
`delta_voto_exit_total_pct` sobre el bloque largo (suma de
`delta_voto_exit_ausentismo_pct` + `delta_voto_exit_blanco_nulo_pct`, D22),
features de trayectoria trimestral `_trim`.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import pandas as pd

general_path = "/workspaces/analisis-politica-economia/"
data_path = f"{{general_path}}data/tfi_data/"
sys.path.insert(0, f"{{general_path}}/src")
from ml_models.cargar_panel import cargar_panel, columnas_candidatas
from ml_models.lasso import *
NIVELES = ["municipal", "provincial", "nacional"]
paneles = {{nivel: cargar_panel(nivel, f"{{data_path}}panel_ventanas_bieleccion.csv") for nivel in NIVELES}}

for nivel, df in paneles.items():
    print(f"{{nivel}}: {{df.shape[0]}} filas x {{df.shape[1]}} columnas")

In [ ]:
# delta_voto_exit_total_pct no es columna del panel (D22, misma
# redundancia algebraica que en ventana_t-1/) -- se reconstruye acá porque
# este notebook la usa como hipótesis propia (exit total, no sus partes).
for nivel in NIVELES:
    paneles[nivel]["delta_voto_exit_total_pct"] = (
        paneles[nivel]["delta_voto_exit_ausentismo_pct"] + paneles[nivel]["delta_voto_exit_blanco_nulo_pct"]
    )

In [ ]:
cols_vc_por_nivel = {}
corr_por_nivel = {}
for nivel in NIVELES:
    df = paneles[nivel]
    cols_vc = columnas_candidatas(df, excluir_adicional=["delta_voto_exit_total_pct"])

    corr = df[cols_vc].corr(method="pearson")  # pairwise, ignora NaN automáticamente
    cols_vc_por_nivel[nivel] = cols_vc
    corr_por_nivel[nivel] = corr
for nivel in NIVELES:
    print(f"{nivel}: {len(cols_vc_por_nivel[nivel])} variables candidatas, N={len(paneles[nivel])}")

### Paso 2 — Sub-selección: colapsar clusters redundantes

Mismo criterio que su equivalente en `ventana_t-1/` (umbral `|r| >= 0.90`,
single-linkage, desempate por sufijo `_nivel_trim` preferido sobre
`_final_trim`/`_pendiente_trim`/`_volatilidad_trim`, después por prioridad
teórica de la variable) -- acá aplicado sobre la trayectoria trimestral del
bloque largo (t-2 -> t) en vez de la mensual de la ventana corta.

In [ ]:
UMBRAL_REDUNDANCIA = 0.90
PRIORIDAD_TEORICA = ["ipc", "desocupacion", "icg", "icc", "salario_real", "tc_oficial", "reservas", "resultado_fiscal", "emae"]
ORDEN_SUFIJO = ["_nivel_trim", "_final_trim", "_pendiente_trim", "_volatilidad_trim"]

clusters_por_nivel = {}
columnas_finales_por_nivel = {}

for nivel in NIVELES:
    df = paneles[nivel]
    cols_vc = cols_vc_por_nivel[nivel]
    corr = corr_por_nivel[nivel]
    print(f"\n\nNivel: {nivel}")
    clusters_por_nivel[nivel] = encontrar_redundantes(corr, UMBRAL_REDUNDANCIA)
    columnas_finales_por_nivel[nivel] = [elegir_representante(cl, df, ORDEN_SUFIJO, PRIORIDAD_TEORICA) if len(cl) > 1 else next(iter(cl)) for cl in clusters_por_nivel[nivel]]

    print(f"De {len(cols_vc)} columnas candidatas, quedan {len(columnas_finales_por_nivel[nivel])} tras colapsar clusters (umbral={UMBRAL_REDUNDANCIA})\n")
    for cl in clusters_por_nivel[nivel]:
        if len(cl) > 1:
            elegido = elegir_representante(cl, df, ORDEN_SUFIJO, PRIORIDAD_TEORICA)
            print(f"cluster ({len(cl)}): {sorted(cl)} -> queda: {elegido}")

In [ ]:
datos_final = {}
for nivel in NIVELES:
    X, y = construir_Xy_final(nivel, columnas_finales_por_nivel[nivel], paneles, target="delta_voto_exit_total_pct")
    datos_final[nivel] = (X, y)
    print(f"{nivel}: N={len(y)}, P={X.shape[1]}")

### Paso 3 — LASSO por coordinate descent (implementación propia)

Formulación: `(1/2n)·‖y - Xβ‖² + α·‖β‖₁` (convención sklearn/glmnet). `X`
estandarizada a mano (`ddof=0`), `y` centrada; intercepto = `media(y)`, no
se penaliza.

In [ ]:
for nivel in NIVELES:
    X_df, y_ser = datos_final[nivel]
    X_std, medias, desvios = estandarizar(X_df)
    y_centrado = y_ser.values - y_ser.mean()
    n = len(y_centrado)

    alpha_prueba = 1.0
    beta_manual = lasso_coordinate_descent(X_std, y_centrado, alpha_prueba)

    kkt = verificar_kkt(X_std, y_centrado, beta_manual, alpha_prueba, n)
    print(f"{nivel} - KKT:", kkt)

    beta_alpha_cero = lasso_coordinate_descent(X_std, y_centrado, alpha=0.0, max_iter=5000)
    beta_ols, *_ = np.linalg.lstsq(X_std, y_centrado, rcond=None)
    print(f"{nivel} - Máxima diferencia vs. OLS (alpha=0):", np.max(np.abs(beta_alpha_cero - beta_ols)))

### Paso 4 — Grilla de alpha + LOO-CV manual

In [ ]:
resultados_cv = {}
for nivel in NIVELES:
    X_df, y_ser = datos_final[nivel]
    resultados_cv[nivel] = lasso_loocv_manual(X_df, y_ser, factor_extension=3.0)
    r = resultados_cv[nivel]
    print(f"{nivel}: alpha_min={r['alpha_min']:.4f}  alpha_1se={r['alpha_1se']:.4f}  (techo grilla={r['alphas'][-1]:.4f})")

In [ ]:
for nivel in NIVELES:
    X_df, y_ser = datos_final[nivel]
    print(f"--- {nivel} ---")
    print(verificar_saturacion(X_df, y_ser, factores=[1, 3, 10]))
    print()

### Paso 5 — Ajuste final: coeficientes y mejora sobre baseline trivial

`baseline_trivial_loocv`: MSE en LOO de predecir el promedio de los demás
puntos. `mse_en_alpha`: mismo esquema de LOO, para un alpha puntual.

In [ ]:
resumen = []
coeficientes_min, coeficientes_1se = {}, {}

for nivel in NIVELES:
    X_df, y_ser = datos_final[nivel]
    r = resultados_cv[nivel]

    base = baseline_trivial_loocv(y_ser)
    mse_min = mse_en_alpha(X_df, y_ser, r["alpha_min"])
    mse_1se = mse_en_alpha(X_df, y_ser, r["alpha_1se"])

    resumen.append({
        "nivel": nivel,
        "baseline_mse": base,
        "mejora_alpha_min_%": 100 * (1 - mse_min / base),
        "mejora_alpha_1se_%": 100 * (1 - mse_1se / base),
    })

    coeficientes_min[nivel] = ajustar_final(X_df, y_ser, r["alpha_min"])
    coeficientes_1se[nivel] = ajustar_final(X_df, y_ser, r["alpha_1se"])

tabla_resumen = pd.DataFrame(resumen).set_index("nivel")
print(tabla_resumen)

In [ ]:
tabla_coef_min = pd.DataFrame(coeficientes_min)
tabla_coef_min = tabla_coef_min[(tabla_coef_min != 0).any(axis=1)]
print("Coeficientes distintos de cero (alpha_min):")
tabla_coef_min

### Paso 6 — Chequeo de estabilidad (leave-one-transition-out), los tres niveles

In [ ]:
for nivel in NIVELES:
    X_df, y_ser = datos_final[nivel]
    alpha = resultados_cv[nivel]["alpha_1se"]
    resultado = estabilidad_seleccion(nivel, alpha, paneles[nivel], columnas_finales_por_nivel[nivel], "delta_voto_exit_total_pct", X_df, y_ser)
    sobrevivientes = resultado.loc[:, (resultado != 0).any(axis=0)]

    print(f"--- {nivel} (alpha_1se={alpha:.3f}) ---")
    if sobrevivientes.empty:
        print("Ninguna variable sobrevive en ninguna de las corridas leave-one-transition-out.\n")
    else:
        print(sobrevivientes)
        print()